In [ ]:
from pathlib import Path
import pandas as pd
import shutil

# ----------------------------
# Paths
# ----------------------------
images_folder = Path("/data/colon_cancer/Classifier/Decathlon/raw_splitted/imagesTs")  
labels_folder = Path("/data/colon_cancer/Classifier/Decathlon/raw_splitted/labelsTs")  
output_csv = Path("/data/colon_cancer/Classifier/Decathlon/labels.csv") 
"""
for img_path in images_folder.glob("*0000.nii.gz"): 
    # Extract number from filename
    number = img_path.name.replace("_0000.nii.gz", "")
    number_int = int(number)  # convert to int to remove leading zeros
    
    new_name = f"._colon_{number_int}.nii.gz"  # e.g., 12_0000.nii
    new_path = images_folder / new_name
    shutil.move(str(img_path), str(new_path))
"""


# ----------------------------
# Rename images
# ----------------------------
renamed_images = []

for img_path in images_folder.glob("colon_*.nii.gz"):
    # Extract number from filename
    number = img_path.name.replace("colon_", "").replace(".nii.gz", "")  
    number_int = int(number)  # convert to int to remove leading zeros
    
    new_name = f"{number_int}_0000.nii.gz"  # e.g., 12_0000.nii
    new_path = images_folder / new_name
    shutil.move(str(img_path), str(new_path))
    
    renamed_images.append(new_path)

# ----------------------------
# Rename labels
# ----------------------------
for lbl_path in labels_folder.glob("colon_*.nii.gz"):
    number = lbl_path.name.replace("colon_", "").replace(".nii.gz", "")  # '001' from 'colon_001'
    number_int = int(number)  # convert to int
    new_name = f"{number_int}.nii.gz"
    new_path = labels_folder / new_name
    shutil.move(str(lbl_path), str(new_path))

# ----------------------------
# Generate CSV
# ----------------------------
data = []

for img_path in sorted(images_folder.glob("*_0000.nii.gz")):
    uid = img_path.stem.split("_")[0]  # get the UID (number before '_0000')
    data.append({
        "UID": uid,
        "img_path": str(img_path),
        "target": 1,
        "Split": "test",
        "Fold": 0
    })

df = pd.DataFrame(data, columns=["UID", "img_path", "target", "Split", "Fold"])
df.to_csv(output_csv, index=False)
print(f"CSV saved to {output_csv}")


In [2]:
#####################  discard T1 samples #####################################
import os
import shutil

# =========================
# CONFIG
# =========================

src_images_dir = "/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/imagesTr"
src_labels_dir = "/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/labelsTr"

dst_images_dir = "/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/images_excluded"
dst_labels_dir = "/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/labels_excluded"

uids = [
    "525",
    "547",
    "656",
    "551",
    "163",
    "569",
    "529",
    "556",
    "563",
    "527",
    "566",
    "586",
    "571",
    "580",
    "562",
    "525",
    "548",
    "275",
    "510",
    "547",
    "521",
    "575",
    "531",
    "589",
    "577",
    "514",
    "541",
    "553",
    "539",
    "561",
    "505",
    "540",
    "532",
    "517",
    "522",
    "512",
    "593",
    "504",
    "516",
    "587",
]

# =========================
# SETUP
# =========================

os.makedirs(dst_images_dir, exist_ok=True)
os.makedirs(dst_labels_dir, exist_ok=True)

# =========================
# MOVE FILES
# =========================

for uid in uids:
    img_name = f"{uid}_0000.nii.gz"
    lbl_name = f"{uid}.nii.gz"

    src_img = os.path.join(src_images_dir, img_name)
    src_lbl = os.path.join(src_labels_dir, lbl_name)

    dst_img = os.path.join(dst_images_dir, img_name)
    dst_lbl = os.path.join(dst_labels_dir, lbl_name)

    print(f"\nProcessing UID: {uid}")

    # --- Image ---
    if os.path.exists(src_img):
        shutil.move(src_img, dst_img)
        print(f"  ✓ Moved image → {dst_img}")
    else:
        print(f"  ⚠ Image not found: {src_img}")

    # --- Label ---
    if os.path.exists(src_lbl):
        shutil.move(src_lbl, dst_lbl)
        print(f"  ✓ Moved label → {dst_lbl}")
    else:
        print(f"  ⚠ Label not found: {src_lbl}")

print("\nDone.")



Processing UID: 525
  ✓ Moved image → /data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/images_excluded/525_0000.nii.gz
  ✓ Moved label → /data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/labels_excluded/525.nii.gz

Processing UID: 547
  ✓ Moved image → /data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/images_excluded/547_0000.nii.gz
  ✓ Moved label → /data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/labels_excluded/547.nii.gz

Processing UID: 656
  ✓ Moved image → /data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/images_excluded/656_0000.nii.gz
  ✓ Moved label → /data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/labels_excluded/656.nii.gz

Processing UID: 551
  ✓ Moved image → /data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/images_excluded/551_0000.nii.gz
  ✓ Moved label → /data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/labels_excluded/551.nii.gz

Processing UID: 163
  ✓ Moved image → /data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/im

In [ ]:
################################## clean labels csv #####################################
import pandas as pd

csv_path = "/data/colon_cancer/Classifier/ColonCancer/labels.csv"
output_csv_path = "/data/colon_cancer/Classifier/ColonCancer/labels_cleaned.csv"

# UIDs to remove (convert once, remove duplicates)
uids = [
    525, 547, 656, 551, 163, 569, 529, 556, 563, 527, 566,
    586, 571, 580, 562, 548, 275, 510, 521, 575, 531, 589,
    577, 514, 541, 553, 539, 561, 505, 540, 532, 517, 522,
    512, 593, 504, 516, 587
]
uids = set(uids)  # deduplicate + faster lookup

# Load CSV
df = pd.read_csv(csv_path)

# Normalize UID column
df["UID"] = df["UID"].astype(str).str.strip().astype(int)

before = len(df)

# Debug: check overlap
overlap = set(df["UID"]).intersection(uids)
print(f"UIDs found in CSV to remove: {sorted(overlap)}")

# Remove rows
df_clean = df[~df["UID"].isin(uids)]

after = len(df)

# Save cleaned CSV
df_clean.to_csv(output_csv_path, index=False)

print(f"Removed {before - after} rows")
print(f"Saved cleaned CSV to: {output_csv_path}")


In [ ]:
########################## split dataset ###############################
import pandas as pd
import numpy as np

# -------------------------
# Config
# -------------------------
INPUT_CSV = "/data/colon_cancer/Classifier/ColonCancer/labels_cleaned.csv"
OUTPUT_CSV = "/data/colon_cancer/Classifier/ColonCancer/splits_cleaned.csv"

N_TRAIN = 600
N_VAL   = 118
N_TEST  = 77

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# -------------------------
# Load data
# -------------------------
df = pd.read_csv(INPUT_CSV)

assert len(df) == N_TRAIN + N_VAL + N_TEST, "Split sizes do not sum to dataset size!"

# -------------------------
# Compute class proportions
# -------------------------
class_counts = df["target"].value_counts().sort_index()
total = len(df)

class_ratios = class_counts / total

# Samples per class per split
def split_counts(n_total):
    counts = (class_ratios * n_total).round().astype(int)
    # fix rounding errors
    diff = n_total - counts.sum()
    if diff != 0:
        counts.iloc[0] += diff
    return counts

train_counts = split_counts(N_TRAIN)
val_counts   = split_counts(N_VAL)
test_counts  = split_counts(N_TEST)

# -------------------------
# Perform stratified split
# -------------------------
df["split"] = None
remaining_idx = []

for cls in class_counts.index:
    cls_df = df[df["target"] == cls].sample(frac=1, random_state=RANDOM_SEED)

    n_train = train_counts[cls]
    n_val   = val_counts[cls]
    n_test  = test_counts[cls]

    train_idx = cls_df.iloc[:n_train].index
    val_idx   = cls_df.iloc[n_train:n_train + n_val].index
    test_idx  = cls_df.iloc[n_train + n_val:n_train + n_val + n_test].index

    df.loc[train_idx, "split"] = "train"
    df.loc[val_idx, "split"]   = "val"
    df.loc[test_idx, "split"]  = "test"

# -------------------------
# Sanity checks
# -------------------------
print("\nSplit sizes:")
print(df["split"].value_counts())

print("\nClass balance per split:")
print(df.groupby(["split", "target"]).size().unstack())

assert df["split"].isna().sum() == 0, "Some samples were not assigned a split!"

# -------------------------
# Save
# -------------------------
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved splits to {OUTPUT_CSV}")



Split sizes:
split
train    600
val      118
test      77
Name: count, dtype: int64

Class balance per split:
target    0    1
split           
test     32   45
train   251  349
val      49   69

Saved splits to /data/colon_cancer/Classifier/ColonCancer/splits_cleaned.csv


In [ ]:
import os
from pathlib import Path
import SimpleITK as sitk
import pydicom


# ============================
# CONFIG
# ============================

ROOT_DIR = Path("/data/colon_cancer/normal/NomalCT")
OUT_DIR = Path("/data/colon_cancer/normal/nifti")

OUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================
# HELPERS
# ============================

def looks_like_dicom(path):
    """Very permissive DICOM check (Sectra-safe)."""
    try:
        pydicom.dcmread(path, stop_before_pixels=True, force=True)
        return True
    except:
        return False


def is_dicom_folder(folder):
    """Detect folders that actually contain DICOM slices."""
    if not folder.is_dir():
        return False

    checked = 0
    for f in folder.iterdir():
        if f.is_file():
            checked += 1
            if looks_like_dicom(f):
                return True
        if checked >= 5:   # do not scan entire folder
            break
    return False


def dicom_folder_to_nifti(dicom_dir, output_path):
    reader = sitk.ImageSeriesReader()
    series_ids = reader.GetGDCMSeriesIDs(dicom_dir)

    if not series_ids:
        raise RuntimeError(f"No DICOM series found in {dicom_dir}")

    series_files = reader.GetGDCMSeriesFileNames(dicom_dir, series_ids[0])
    reader.SetFileNames(series_files)

    image = reader.Execute()
    sitk.WriteImage(image, output_path)


# ============================
# MAIN
# ============================

for patient_dir in sorted(ROOT_DIR.iterdir()):
    if not patient_dir.is_dir():
        continue

    patient_id = patient_dir.name
    dicom_root = patient_dir / "DICOM"

    if not dicom_root.exists():
        print(f"[SKIP] {patient_id}: No DICOM folder")
        continue

    print(f"\nProcessing patient {patient_id}")

    found = False

    for root, dirs, files in os.walk(dicom_root):
        root = Path(root)

        if files and is_dicom_folder(root):
            out_path = OUT_DIR / f"{patient_id}.nii.gz"
            print(f"  → Found DICOM series in {root}")
            dicom_folder_to_nifti(str(root), str(out_path))
            found = True
            break  # one scan per patient

    if not found:
        print(f"  ❌ No DICOM series found for patient {patient_id}")

print("\nDone.")
#8,9,28,33

In [ ]:
from pathlib import Path

nifti_dir = Path("/data/colon_cancer/normal/nifti")

for nifti in nifti_dir.iterdir():
    if nifti.suffix == ".gz" and nifti.name.endswith(".nii.gz"):
        stem = nifti.name[:-7]   # remove .nii.gz
        if stem.endswith("_0000"):
            continue

        new_name = f"{stem}_0000.nii.gz"
        new_path = nifti_dir / new_name

        if new_path.exists():
            print(f"[SKIP] {new_name} already exists")
            continue

        nifti.rename(new_path)
        print(f"Renamed: {nifti.name} → {new_name}")

    elif nifti.suffix == ".nii":
        stem = nifti.stem
        if stem.endswith("_0000"):
            continue

        new_name = f"{stem}_0000.nii"
        new_path = nifti_dir / new_name

        if new_path.exists():
            print(f"[SKIP] {new_name} already exists")
            continue

        nifti.rename(new_path)
        print(f"Renamed: {nifti.name} → {new_name}")


Renamed: 45.nii.gz → 45_0000.nii.gz
Renamed: 48.nii.gz → 48_0000.nii.gz
Renamed: 1.nii.gz → 1_0000.nii.gz
Renamed: 50.nii.gz → 50_0000.nii.gz
Renamed: 23.nii.gz → 23_0000.nii.gz
Renamed: 6.nii.gz → 6_0000.nii.gz
Renamed: 27.nii.gz → 27_0000.nii.gz
Renamed: 14.nii.gz → 14_0000.nii.gz
Renamed: 18.nii.gz → 18_0000.nii.gz
Renamed: 16.nii.gz → 16_0000.nii.gz
Renamed: 40.nii.gz → 40_0000.nii.gz
Renamed: 34.nii.gz → 34_0000.nii.gz
Renamed: 38.nii.gz → 38_0000.nii.gz
Renamed: 21.nii.gz → 21_0000.nii.gz
Renamed: 10.nii.gz → 10_0000.nii.gz
Renamed: 30.nii.gz → 30_0000.nii.gz
Renamed: 46.nii.gz → 46_0000.nii.gz
Renamed: 26.nii.gz → 26_0000.nii.gz
Renamed: 9.nii.gz → 9_0000.nii.gz
Renamed: 2.nii.gz → 2_0000.nii.gz
Renamed: 15.nii.gz → 15_0000.nii.gz
Renamed: 28.nii.gz → 28_0000.nii.gz
Renamed: 29.nii.gz → 29_0000.nii.gz
Renamed: 5.nii.gz → 5_0000.nii.gz
Renamed: 36.nii.gz → 36_0000.nii.gz
Renamed: 3.nii.gz → 3_0000.nii.gz
Renamed: 24.nii.gz → 24_0000.nii.gz
Renamed: 7.nii.gz → 7_0000.nii.gz
Rename

In [7]:
##################### move test samples ###############################import os
import shutil
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# Paths (EDIT THESE)
# --------------------------------------------------
splits_csv = "/data/colon_cancer/Classifier/ColonCancer/splits_cleaned.csv"

images_src = Path("/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/imagesTr")
labels_src = Path("/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/labelsTr")

images_dst = Path("/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/imagesTs")
labels_dst = Path("/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/labelsTs")

# Create destination folders
images_dst.mkdir(parents=True, exist_ok=True)
labels_dst.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# Load CSV
# --------------------------------------------------
df = pd.read_csv(splits_csv)

# Normalize columns
df["UID"] = df["UID"].astype(str)
df["split"] = df["split"].str.lower()

# Select test set
test_uids = df[df["split"] == "test"]["UID"].tolist()

print(f"Found {len(test_uids)} test cases")

# --------------------------------------------------
# Move files
# --------------------------------------------------
moved_images = 0
moved_labels = 0

for uid in test_uids:
    img_name = f"{uid}_0000.nii.gz"
    lbl_name = f"{uid}.nii.gz"

    img_src = images_src / img_name
    lbl_src = labels_src / lbl_name

    img_dst = images_dst / img_name
    lbl_dst = labels_dst / lbl_name

    # Move image
    if img_src.exists():
        shutil.move(str(img_src), str(img_dst))
        moved_images += 1
        print(f"✓ Moved image: {img_name}")
    else:
        print(f"⚠ Image missing: {img_name}")

    # Move label
    if lbl_src.exists():
        shutil.move(str(lbl_src), str(lbl_dst))
        moved_labels += 1
        print(f"✓ Moved label: {lbl_name}")
    else:
        print(f"⚠ Label missing: {lbl_name}")

# --------------------------------------------------
# Summary
# --------------------------------------------------
print("\n=== Summary ===")
print(f"Images moved: {moved_images}")
print(f"Labels moved: {moved_labels}")
print("Done ✅")

Found 77 test cases
✓ Moved image: 334_0000.nii.gz
✓ Moved label: 334.nii.gz
✓ Moved image: 620_0000.nii.gz
✓ Moved label: 620.nii.gz
✓ Moved image: 489_0000.nii.gz
✓ Moved label: 489.nii.gz
✓ Moved image: 761_0000.nii.gz
✓ Moved label: 761.nii.gz
✓ Moved image: 31_0000.nii.gz
✓ Moved label: 31.nii.gz
✓ Moved image: 172_0000.nii.gz
✓ Moved label: 172.nii.gz
✓ Moved image: 357_0000.nii.gz
✓ Moved label: 357.nii.gz
✓ Moved image: 78_0000.nii.gz
✓ Moved label: 78.nii.gz
✓ Moved image: 33_0000.nii.gz
✓ Moved label: 33.nii.gz
✓ Moved image: 795_0000.nii.gz
✓ Moved label: 795.nii.gz
✓ Moved image: 148_0000.nii.gz
✓ Moved label: 148.nii.gz
✓ Moved image: 265_0000.nii.gz
✓ Moved label: 265.nii.gz
✓ Moved image: 628_0000.nii.gz
✓ Moved label: 628.nii.gz
✓ Moved image: 346_0000.nii.gz
✓ Moved label: 346.nii.gz
✓ Moved image: 74_0000.nii.gz
✓ Moved label: 74.nii.gz
✓ Moved image: 728_0000.nii.gz
✓ Moved label: 728.nii.gz
✓ Moved image: 176_0000.nii.gz
✓ Moved label: 176.nii.gz
✓ Moved image: 499_